# PA1 — Task 2 v2 (Unsupervised Domain Adaptation on PACS)

**What changed from v1.** Gradient clipping (global norm 1.0) is now applied to **every** method,
including Source-only. Without it, DANN and CDAN diverged in the first epoch: the gradient-reversal
layer maximises a domain loss that has no upper limit, so features and discriminator weights grew
without bound and the classifier collapsed to chance. Clipping bounds the step size without changing
any objective, architecture or the handout's optimiser settings. The pre-clipping gradient norm is
logged so you can report how often clipping was active.

**Before you start**
1. Upload `pa1-task2-v2.zip` to the top level of **My Drive**.
2. Your repository is at **My Drive/PA1** (change `REPO` in A3 if not).
3. **Runtime → Change runtime type → T4 GPU → Save**.

**How this notebook is organised**
- **Part A (setup)** runs every session. A3 replaces the Task 2 code, A3b archives the old diverged runs.
- **Part B (training)** is one cell per run; finished runs are skipped, so you can split this over sessions.
  Budget roughly 15–45 minutes per run on a T4, six runs in total.
- **Part C** is the only place Sketch labels are used.
- **Part D** packages everything for GitHub.

All six runs must be re-done, because every method has to go through an identical pipeline.
**Task 3 will load the new** `task2/results/runs/source_only/best.pt`, so finish this before Task 3.

## Part A — setup (run every session)

In [ ]:
# A1. Confirm a GPU is attached.
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# A2. Mount Google Drive.
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# A3. Replace the Task 2 code with v2.
# `-o` overwrites code and configs on purpose. Your results, splits and Task 1 files are untouched:
# the zip contains no results and no shared/splits. Your own task2/README.md is backed up first.
import os, shutil
DRIVE = '/content/drive/MyDrive'
REPO = f'{DRIVE}/PA1'                      # <- change if your repo folder has another name
ZIP = f'{DRIVE}/pa1-task2-v2.zip'
assert os.path.isdir(REPO), f'Repository not found at {REPO}'
assert os.path.exists(ZIP), f'Upload pa1-task2-v2.zip to My Drive first (expected {ZIP})'
if os.path.exists(f'{REPO}/task2/README.md') and not os.path.exists(f'{REPO}/task2/README_yours.md'):
    shutil.copy(f'{REPO}/task2/README.md', f'{REPO}/task2/README_yours.md')   # keeps any notes you added
!unzip -q -o "{ZIP}" -d "{REPO}"
%cd {REPO}
!sed -n '/grad_clip/,+2p' task2/configs/base.yaml     # must print grad_clip: 1.0
!ls

In [ ]:
# A3b. Archive the old (diverged) runs once, so Part B starts from scratch but the evidence is kept.
import json
if os.path.isdir('task2/results/runs') and not os.path.isdir('task2/results/runs_noclip'):
    shutil.move('task2/results/runs', 'task2/results/runs_noclip')
    print('old runs -> task2/results/runs_noclip')
elif os.path.isdir('task2/results/runs_noclip'):
    print('old runs already archived in task2/results/runs_noclip')
else:
    print('no previous runs found -- nothing to archive')

In [ ]:
# A4. Helper: run a command, stream its output, stop the notebook if it fails.
import subprocess
def run(cmd):
    print('$', cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='')
    if p.wait() != 0:
        raise RuntimeError(f'Command failed (exit code {p.returncode}): {cmd}')

In [ ]:
# A5. PACS on Colab's local disk (reading ~10k small files from Drive is very slow).
# Restores from your My Drive/datasets/PACS.zip backup if it exists; otherwise downloads
# (DomainBed's Google Drive link, GitHub mirror as fallback) and then makes that backup.
import glob
PACS = '/content/data/PACS'
DRIVE_ZIP = f'{DRIVE}/datasets/PACS.zip'
EXPECTED = {'photo': 1670, 'art_painting': 2048, 'cartoon': 2344, 'sketch': 3929}

if not os.path.isdir(f'{PACS}/sketch'):
    os.makedirs('/content/data', exist_ok=True)
    if os.path.exists(DRIVE_ZIP):
        print('Restoring PACS from Drive backup ...')
        run(f'unzip -q "{DRIVE_ZIP}" -d /content/data')
    else:
        try:
            run('python -m shared.download_pacs --root /content/data')
        except RuntimeError:
            print('Google Drive download failed -- trying a GitHub mirror of PACS ...')
            run('git clone -q --depth 1 https://github.com/MachineLearning2020/Homework3-PACS /content/pacs_mirror')
            hits = [os.path.dirname(p) for p in glob.glob('/content/pacs_mirror/**/sketch', recursive=True)
                    if all(os.path.isdir(os.path.join(os.path.dirname(p), d)) for d in EXPECTED)]
            assert hits, 'Could not find the four PACS domain folders in the mirror'
            shutil.move(hits[0], PACS)
        os.makedirs(f'{DRIVE}/datasets', exist_ok=True)
        print('Backing up PACS to Drive ...')
        run(f'cd /content/data && zip -qr "{DRIVE_ZIP}" PACS')

for d, n in EXPECTED.items():
    got = len([p for p in glob.glob(f'{PACS}/{d}/*/*') if p.lower().endswith(('.jpg', '.jpeg', '.png'))])
    print(f'{d:13s} {got:5d} images  (expected {n})' + ('' if got == n else '   <-- MISMATCH'))

In [ ]:
# A6. The shared PACS split (seed 6304). Already created in your first session; reused unchanged,
# so the re-runs use exactly the same images as before.
if not os.path.exists('shared/splits/pacs_sources_seed6304.json'):
    run(f'python -m shared.pacs_protocol --root {PACS}')
src = json.load(open('shared/splits/pacs_sources_seed6304.json'))['domains']
for d, s in src.items():
    print(f'{d:13s} train={len(s["train"]):5d}  val={len(s["val"]):4d}')
print(f'{"sketch":13s} all={len(json.load(open("shared/splits/pacs_sketch_target.json"))["items"])} (unlabelled during training)')

## Part B — training (all six runs, with clipping)

Keep `STUDY = 'dan'` unless you deliberately want the DANN study instead.

While each run prints, the numbers to watch are:
- `loss_cls` should fall from about 1.9 and stay well below it. If it sits at ~1.95, the classifier has collapsed.
- `loss_dom` should hover near 0.69 (ln 2) once the discriminator is confused, not in the hundreds.
- `disc_acc` drifts from ~1.0 toward ~0.5 as alpha grows.
- `grad_norm` is the pre-clipping norm; values above 1.0 mean clipping was active on that epoch's average.

In [ ]:
# B0. Run list + training helper.
STUDY = 'dan'     # 'dan'  -> lambda_MMD in {0.1, 1, 10}
                  # 'dann' -> maximum gradient-reversal strength in {0.25, 0.5, 1}
COMMON = f'--set data_root={PACS}'
RUNS = [('source_only', 'source_only'), ('dan', 'dan'), ('dann', 'dann'), ('cdan', 'cdan')]
RUNS += ([('study_dan_lambda0.1', 'dan_lambda0.1'), ('study_dan_lambda10', 'dan_lambda10')] if STUDY == 'dan' else
         [('study_dann_alpha0.25', 'dann_alpha0.25'), ('study_dann_alpha0.5', 'dann_alpha0.5')])

def train(config, run_name):
    if os.path.exists(f'task2/results/runs/{run_name}/summary.json'):
        print(f'{run_name}: already finished -- skipping')
        return
    run(f'python -m task2.train --config task2/configs/{config}.yaml {COMMON}')

for c, r in RUNS:
    print(f'{"done   " if os.path.exists(f"task2/results/runs/{r}/summary.json") else "pending"}  {r}')

In [ ]:
# B1. Source-only ERM (also Task 3's ERM baseline)
train('source_only', 'source_only')

In [ ]:
# B2. DAN (MMD alignment, lambda = 1)
train('dan', 'dan')

In [ ]:
# B3. DANN (adversarial alignment)
train('dann', 'dann')

In [ ]:
# B4. CDAN (class-conditional adversarial alignment)
train('cdan', 'cdan')

In [ ]:
# B5 + B6. Controlled study: the two extra settings (the middle setting is the main run above)
for c, r in RUNS[4:]:
    train(c, r)

In [ ]:
# B7. Training health check (source information only -- safe to look at).
# final_loss_cls near 1.95 (= ln 7) means that run collapsed to chance; clipped_epochs counts
# epochs whose average pre-clipping gradient norm exceeded the 1.0 threshold.
import pandas as pd
rows = []
for c, r in RUNS:
    p, h = f'task2/results/runs/{r}/summary.json', f'task2/results/runs/{r}/history.json'
    if not os.path.exists(p):
        rows.append({'run': r}); continue
    s, hist = json.load(open(p)), json.load(open(h))
    gn = [e.get('grad_norm', float('nan')) for e in hist]
    rows.append({'run': r, 'epochs_run': s['epochs_run'],
                 'best_mean_src_val_macro_f1': round(s['best_mean_macro_f1'], 2),
                 'final_loss_cls': round(hist[-1]['loss_cls'], 3),
                 'max_grad_norm': round(max(gn), 2) if gn else None,
                 'clipped_epochs': sum(g > 1.0 for g in gn)})
display(pd.DataFrame(rows))

## Part C — final evaluation (uses Sketch labels)

Run only once every run in B7 looks healthy and your hypotheses are written down.
From here on, nothing you see may change any Task 2 or Task 3 setting.

In [ ]:
# C1. Final evaluation: main comparison + controlled study.
missing = [r for _, r in RUNS if not os.path.exists(f'task2/results/runs/{r}/summary.json')]
assert not missing, f'Unfinished runs: {missing} -- finish Part B first'
run('python -m task2.evaluate_final')
run(f'python -m task2.evaluate_final --study {STUDY}')

In [ ]:
# C2. Look at the results.
from IPython.display import Image
F = 'task2/results/final'
for f in ['table_main.csv', f'table_study_{STUDY}.csv', 'per_class_target_acc.csv']:
    print('\n==', f)
    display(pd.read_csv(f'{F}/{f}').round(2))
print('\n== class_analysis.json')
print(open(f'{F}/class_analysis.json').read())
for p in sorted(glob.glob(f'{F}/figures/*.png')):
    print(p)
    display(Image(p))

In [ ]:
# C3. OPTIONAL, analysis only: class shares per domain (this reads Sketch labels, so run it
# only now that everything is locked). Big differences in the sketch column are the setting for
# Zhao et al.'s warning about aligning domains whose label distributions differ.
import collections
from shared.pacs import CLASSES
counts = {d: collections.Counter(y for _, y in s['train'] + s['val']) for d, s in src.items()}
counts['sketch'] = collections.Counter(y for _, y in json.load(open('shared/splits/pacs_sketch_target.json'))['items'])
df = pd.DataFrame(counts).sort_index().rename(index=dict(enumerate(CLASSES)))
print('images per class'); display(df)
print('class share (%)'); display((100 * df / df.sum()).round(1))

## Part D — package for GitHub

In [ ]:
# D1. Zip exactly what git would commit (checkpoints, data and caches excluded by .gitignore).
!rm -rf /tmp/gitcheck && git init -q /tmp/gitcheck
!git --git-dir=/tmp/gitcheck/.git --work-tree=. ls-files --others --exclude-standard > /tmp/commit_list.txt
!echo "files: $(wc -l < /tmp/commit_list.txt)" && du -ch $(cat /tmp/commit_list.txt) | tail -1
!grep -c "^shared/splits/" /tmp/commit_list.txt   # should print 2
!rm -f "{DRIVE}/PA1_github.zip" && zip -q -@ "{DRIVE}/PA1_github.zip" < /tmp/commit_list.txt
print('Saved My Drive/PA1_github.zip')

## If something goes wrong
- **Copy the full error** (last ~30 lines of the cell output) and send it to Claude.
- **A run still diverges** (`loss_cls` ~1.95, or losses in the hundreds): stop, send me its B7 row and the last ~15 printed epochs. The next diagnostic is `study_dann_alpha0.25.yaml`, which tests whether weaker alignment pressure is stable.
- **`FileNotFoundError: No images under ...`:** check the counts printed by A5.
- **`CUDA out of memory`:** Runtime → Restart session, then re-run Part A. Don't change batch sizes; the handout fixes them.
- **Want to start one run over?** Delete its folder under `task2/results/runs/` and re-run its cell.